In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


# Week 1: Data Acquisition, Cleaning, and Preprocessing

## Online Retail Dataset

### Virtual Data Science with Python Trainee Internship

**Objective:**  
To acquire, explore, clean, and preprocess a publicly available dataset using Python, while identifying missing values, inconsistencies, erroneous entries, and outliers and documenting the rationale behind the preprocessing decisions.

## 1. Dataset Acquisition

The dataset selected for this project is the **Online Retail Dataset** from the **UCI Machine Learning Repository**.

### Dataset Source

**Source:** UCI Machine Learning Repository  
**Dataset:** Online Retail  
**URL:** https://archive.ics.uci.edu/dataset/352/online+retail

The dataset contains transactional records from a UK-based online retail business. It includes information about invoices, products, quantities, transaction dates, unit prices, customers, and countries.

The dataset was downloaded in Excel format (`Online Retail.xlsx`) and stored in the project's `data/raw/` directory. The original dataset is preserved without modification so that the preprocessing process remains reproducible.

In [2]:
# Load the Online Retail dataset

file_path = "../data/raw/Online Retail.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [7]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [8]:
df.describe(include="object")

C:\Users\ponna\AppData\Local\Temp\ipykernel_1876\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,InvoiceNo,StockCode,Description,Country
count,541909,541909,540455,541909
unique,25900,4070,4223,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,1114,2313,2369,495478


In [9]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

InvoiceNo: 25900 unique values
StockCode: 4070 unique values
Description: 4223 unique values
Quantity: 722 unique values
InvoiceDate: 23260 unique values
UnitPrice: 1630 unique values
CustomerID: 4372 unique values
Country: 38 unique values


In [10]:
print("Earliest transaction:", df["InvoiceDate"].min())
print("Latest transaction:", df["InvoiceDate"].max())

Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00


In [11]:
df["Country"].value_counts().head(10)

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

In [12]:
print("Quantity:")
print(df["Quantity"].describe())

print("\nUnitPrice:")
print(df["UnitPrice"].describe())

Quantity:
count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

UnitPrice:
count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


## 2. Missing Value Analysis

Missing-value analysis is performed to identify incomplete records and determine the appropriate treatment for each affected column. The decision to remove, retain, or handle missing values will be based on the role of the column and the potential impact on subsequent analysis.

In [13]:
missing_count = df.isnull().sum()

missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

missing_summary

,Missing Values,Missing Percentage
CustomerID,135080,24.926694
Description,1454,0.268311
StockCode,0,0.000000
InvoiceNo,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


### Findings and Treatment of Missing Values

The initial analysis identified missing values in two columns: `CustomerID` and `Description`.

The `CustomerID` column contains 135,080 missing values, representing approximately 24.93% of the dataset. Investigation showed that records with missing CustomerID can still contain valid transaction information such as invoice number, stock code, quantity, invoice date, unit price, and country. Therefore, these records were not removed solely because the CustomerID was missing. The records will be retained, while customer-level analysis will be restricted to transactions with available customer identifiers.

The `Description` column contains 1,454 missing values, representing approximately 0.27% of the dataset. Initial inspection showed that several records with missing descriptions have a UnitPrice of 0.00 and may represent non-standard transaction or adjustment records. Therefore, descriptions will not be filled with arbitrary values. These records will be investigated further during the erroneous-entry and transaction-validity analysis before a final treatment decision is made.

In [14]:
df[df["CustomerID"].isnull()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


In [15]:
df[df["CustomerID"].isnull()][
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "InvoiceDate", "UnitPrice", "Country"]
].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,United Kingdom


In [16]:
df[df["CustomerID"].isnull()]["Country"].value_counts().head(10)

Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
France                66
Israel                47
Portugal              39
Bahrain                2
Name: count, dtype: int64

In [17]:
df[df["CustomerID"].isnull()]["Quantity"].describe()


count    135080.000000
mean          1.995573
std          66.696153
min       -9600.000000
25%           1.000000
50%           1.000000
75%           3.000000
max        5568.000000
Name: Quantity, dtype: float64

## 3. Duplicate Record Analysis

Duplicate records can introduce bias into subsequent analysis by causing certain transactions to be counted more than once. Duplicate rows will therefore be identified and investigated before deciding whether they should be removed.

In [18]:
# Count duplicate records

duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 5268


In [19]:
# Inspect duplicate records

df[df.duplicated(keep=False)].sort_values(
    by=["InvoiceNo", "StockCode"]
).head(20)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [20]:
# Calculate duplicate percentage

duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

Duplicate percentage: 0.97%


### Findings and Treatment of Duplicate Records

The duplicate analysis identified 5,268 duplicate records, representing approximately 0.97% of the original dataset.

The duplicate records were investigated by displaying repeated rows and comparing their values across the available columns. The identified records represent exact duplicate rows, meaning the same transaction information is repeated across all columns.

Because exact duplicate records can cause transactions to be counted more than once and may distort subsequent statistical analysis, these duplicate rows will be removed. Only the first occurrence of each exact duplicate record will be retained.

This treatment reduces redundant records while preserving unique transaction information.

In [21]:
# Store the original row count before removing duplicates

rows_before_duplicates = len(df)

# Remove exact duplicate rows

df = df.drop_duplicates()

rows_after_duplicates = len(df)

duplicates_removed = rows_before_duplicates - rows_after_duplicates

print("Rows before removing duplicates:", rows_before_duplicates)
print("Rows after removing duplicates:", rows_after_duplicates)
print("Duplicates removed:", duplicates_removed)

Rows before removing duplicates: 541909
Rows after removing duplicates: 536641
Duplicates removed: 5268


In [22]:
# Verify that no exact duplicate rows remain

print("Remaining duplicate records:", df.duplicated().sum())

Remaining duplicate records: 0


## 4. Identification of Erroneous and Inconsistent Entries

After addressing exact duplicate records, the dataset was examined for potentially erroneous or inconsistent transaction values. Particular attention was given to Quantity and UnitPrice because the initial statistical analysis revealed negative and unusually large values.

These values will be investigated in their transaction context before any records are removed or modified. This approach helps distinguish genuine business events, such as cancellations or returns, from actual data-entry errors.

In [23]:
negative_quantity_count = (df["Quantity"] < 0).sum()

print("Records with negative Quantity:", negative_quantity_count)
print("Percentage of dataset:", 
      round((negative_quantity_count / len(df)) * 100, 2), "%")

Records with negative Quantity: 10587
Percentage of dataset: 1.97 %


In [24]:
df[df["Quantity"] < 0][
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "InvoiceDate", "UnitPrice",
     "CustomerID", "Country"]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


### Investigation of Negative Quantity Values

The analysis identified 10,587 records with negative quantities, representing approximately 1.97% of the dataset after duplicate removal.

Inspection of sample records shows that many of these transactions have invoice numbers beginning with the letter `C`. These records may represent cancelled transactions rather than simple data-entry errors. Therefore, the relationship between negative quantities and cancelled invoices will be investigated before deciding how these records should be treated.

In [25]:
negative_quantity = df[df["Quantity"] < 0].copy()

negative_quantity["IsCancelledInvoice"] = (
    negative_quantity["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

negative_quantity["IsCancelledInvoice"].value_counts()

IsCancelledInvoice
True     9251
False    1336
Name: count, dtype: int64

In [26]:
cancelled_negative = negative_quantity["IsCancelledInvoice"].sum()
non_cancelled_negative = len(negative_quantity) - cancelled_negative

print("Negative quantity records:", len(negative_quantity))
print("Negative quantity with C invoice:", cancelled_negative)
print("Negative quantity without C invoice:", non_cancelled_negative)

print(
    "Percentage with C invoice:",
    round(cancelled_negative / len(negative_quantity) * 100, 2),
    "%"
)

Negative quantity records: 10587
Negative quantity with C invoice: 9251
Negative quantity without C invoice: 1336
Percentage with C invoice: 87.38 %


In [27]:
cancelled_invoices = df[
    df["InvoiceNo"].astype(str).str.startswith("C")
]

print("Records with cancelled invoices:", len(cancelled_invoices))
print(
    "Cancelled invoices with negative quantity:",
    (cancelled_invoices["Quantity"] < 0).sum()
)

Records with cancelled invoices: 9251
Cancelled invoices with negative quantity: 9251


### Findings from Negative Quantity Investigation

The analysis identified 10,587 records with negative quantities after duplicate records were removed. Of these, 9,251 records (87.38%) have invoice numbers beginning with the letter `C`, which is consistent with the dataset's cancellation transaction pattern.

All 9,251 records associated with cancelled invoices have negative quantities. However, 1,336 negative-quantity records do not have a `C` invoice prefix. These records cannot be classified as cancelled transactions solely from the invoice number and therefore require additional investigation.

The cancelled transactions will be treated separately from potentially erroneous negative-quantity records. The remaining 1,336 records will be examined before a final preprocessing decision is made.

In [28]:
# Investigate negative quantities without a cancelled invoice prefix

non_cancelled_negative = df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
].copy()

print("Non-cancelled negative-quantity records:",
      len(non_cancelled_negative))

non_cancelled_negative.head(20)

Non-cancelled negative-quantity records: 1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


In [29]:
non_cancelled_negative[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,1336.0,1336.0,474,1336.000000,1336,1336.0,0.0,1336
unique,1336.0,1082.0,138,NaN,NaN,NaN,NaN,1
top,536589.0,85175.0,check,NaN,NaN,NaN,NaN,United Kingdom
freq,1.0,5.0,120,NaN,NaN,NaN,NaN,1336
mean,NaN,NaN,NaN,-154.907934,2011-06-15 11:55:00.314371,0.0,NaN,NaN
min,NaN,NaN,NaN,-9600.000000,2010-12-01 16:50:00,0.0,NaN,NaN
25%,NaN,NaN,NaN,-84.000000,2011-03-30 16:40:45,0.0,NaN,NaN
50%,NaN,NaN,NaN,-30.000000,2011-06-08 11:45:30,0.0,NaN,NaN
75%,NaN,NaN,NaN,-8.000000,2011-09-23 14:41:15,0.0,NaN,NaN
max,NaN,NaN,NaN,-1.000000,2011-12-08 15:24:00,0.0,NaN,NaN


In [30]:
non_cancelled_negative["Country"].value_counts().head(10)

Country
United Kingdom    1336
Name: count, dtype: int64

In [31]:
non_cancelled_negative["UnitPrice"].describe()

count    1336.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: UnitPrice, dtype: float64

In [32]:
non_cancelled_negative[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice"
    ]
].head(50)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
2406,536589,21777,NaN,-10,0.0
4347,536764,84952C,NaN,-38,0.0
7188,536996,22712,NaN,-20,0.0
7189,536997,22028,NaN,-20,0.0
7190,536998,85067,NaN,-6,0.0
7192,537000,21414,NaN,-22,0.0
7193,537001,21653,NaN,-6,0.0
7195,537003,85126,NaN,-2,0.0
7196,537004,21814,NaN,-30,0.0
7197,537005,21692,NaN,-70,0.0


### Findings and Treatment of Non-Cancelled Negative Quantities

Further investigation was performed on the 1,336 negative-quantity records that did not have a `C` prefix in their invoice numbers.

All 1,336 records had a UnitPrice of 0.00 and a missing CustomerID. The records were also associated exclusively with the United Kingdom. Many records had missing descriptions, while some contained descriptions such as "check" or "damages".

Based on these combined characteristics, these records do not appear to represent normal sales transactions. Since they contain negative quantities together with zero transaction value and incomplete customer/product information, they were classified as invalid or non-standard records for the cleaned transaction dataset.

Therefore, these 1,336 records will be removed from the cleaned dataset. This decision is based on the combination of multiple data-quality indicators rather than the negative quantity alone.

In [33]:
# Store the count before removing invalid non-cancelled negative records

invalid_negative_count = len(non_cancelled_negative)

print(
    "Non-cancelled negative-quantity records to remove:",
    invalid_negative_count
)

print(
    "Percentage of current dataset:",
    round((invalid_negative_count / len(df)) * 100, 2),
    "%"
)

Non-cancelled negative-quantity records to remove: 1336
Percentage of current dataset: 0.25 %


In [34]:
# Remove non-cancelled negative-quantity records
# that were identified as invalid/non-standard.

df = df.drop(index=non_cancelled_negative.index)

print("Dataset size after removing invalid records:", df.shape)

Dataset size after removing invalid records: (535305, 8)


In [35]:
# Verify that no negative quantities without C invoices remain

remaining_non_cancelled_negative = df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

print(
    "Remaining non-cancelled negative-quantity records:",
    len(remaining_non_cancelled_negative)
)

Remaining non-cancelled negative-quantity records: 0


## 5. UnitPrice Validation

The initial statistical analysis identified unusual values in the `UnitPrice` column, including zero, negative, and extremely high prices. These values will be investigated to determine whether they represent valid business transactions or erroneous/non-standard records before deciding on an appropriate treatment.

In [36]:
zero_price_count = (df["UnitPrice"] == 0).sum()
negative_price_count = (df["UnitPrice"] < 0).sum()

print("Records with UnitPrice = 0:", zero_price_count)
print("Records with UnitPrice < 0:", negative_price_count)

Records with UnitPrice = 0: 1174
Records with UnitPrice < 0: 2


In [37]:
df[df["UnitPrice"] < 0][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [38]:
df[df["UnitPrice"] == 0][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
4348,536765,84952C,NaN,19,2010-12-02 14:43:00,0.0,NaN,United Kingdom


### Findings and Treatment of Negative UnitPrice Values

The analysis identified only 2 records with a negative UnitPrice. Both records have a UnitPrice of -11062.06 and contain the description "Adjust bad debt". The records also have missing CustomerID values.

These records do not represent normal product transactions. Instead, their descriptions indicate accounting-related adjustments. Since a negative unit price is not appropriate for the transaction-level sales dataset being prepared, these 2 records will be removed from the cleaned transaction dataset.

In [39]:
negative_price_records = df[df["UnitPrice"] < 0]

negative_price_count = len(negative_price_records)

print("Negative UnitPrice records to remove:", negative_price_count)

Negative UnitPrice records to remove: 2


In [40]:
df = df[df["UnitPrice"] >= 0].copy()

print("Dataset shape after removing negative UnitPrice records:", df.shape)

Dataset shape after removing negative UnitPrice records: (535303, 8)


In [41]:
print(
    "Remaining negative UnitPrice records:",
    (df["UnitPrice"] < 0).sum()
)

Remaining negative UnitPrice records: 0


In [42]:
zero_price = df[df["UnitPrice"] == 0].copy()

print("Zero UnitPrice records:", len(zero_price))
print("\nQuantity summary:")
print(zero_price["Quantity"].describe())

Zero UnitPrice records: 1174

Quantity summary:
count     1174.000000
mean        61.837308
std        454.442215
min          1.000000
25%          1.000000
50%          3.000000
75%         20.000000
max      12540.000000
Name: Quantity, dtype: float64


In [43]:
print(
    "Zero-price records with missing Description:",
    zero_price["Description"].isnull().sum()
)

print(
    "Zero-price records with missing CustomerID:",
    zero_price["CustomerID"].isnull().sum()
)

Zero-price records with missing Description: 592
Zero-price records with missing CustomerID: 1134


In [44]:
zero_price["Country"].value_counts().head(10)

Country
United Kingdom    1156
EIRE                 4
Netherlands          4
Australia            3
Germany              2
Switzerland          1
Spain                1
RSA                  1
France               1
Norway               1
Name: count, dtype: int64

In [45]:
zero_price["Description"].value_counts(dropna=True).head(20)

Description
check                              39
found                              25
adjustment                         14
FRENCH BLUE METAL DOOR SIGN 1       9
amazon                              8
FRENCH BLUE METAL DOOR SIGN 8       8
Found                               8
FRENCH BLUE METAL DOOR SIGN 4       7
FRENCH BLUE METAL DOOR SIGN No      7
OWL DOORSTOP                        7
FRENCH BLUE METAL DOOR SIGN 3       7
RECIPE BOX PANTRY YELLOW DESIGN     7
Amazon                              7
FRENCH BLUE METAL DOOR SIGN 7       6
FRENCH BLUE METAL DOOR SIGN 5       6
FRENCH BLUE METAL DOOR SIGN 6       6
RED KITCHEN SCALES                  6
?                                   6
Manual                              6
BOX OF 24 COCKTAIL PARASOLS         5
Name: count, dtype: int64

In [46]:
print(
    "Zero-price records with negative Quantity:",
    (zero_price["Quantity"] < 0).sum()
)

Zero-price records with negative Quantity: 0


### Investigation of Zero UnitPrice Values

The dataset contains 1,174 records with a UnitPrice of 0.00. Further investigation found that 1,134 of these records have a missing CustomerID and 592 have a missing Description. None of the zero-price records have a negative Quantity.

Most zero-price records are from the United Kingdom. The available descriptions include terms such as "check", "found", "adjustment", "amazon", and "Manual", in addition to some regular product descriptions.

Because zero price can potentially represent either a non-standard transaction or a legitimate free/promotional item, all zero-price records will not be removed automatically. The smaller group of records with available CustomerID information will be examined before determining the appropriate treatment.

In [47]:
zero_price_with_customer = zero_price[
    zero_price["CustomerID"].notna()
].copy()

print(
    "Zero-price records with CustomerID:",
    len(zero_price_with_customer)
)

zero_price_with_customer[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
]

Zero-price records with CustomerID: 40


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [48]:
zero_price_with_customer["Description"].value_counts(dropna=False).head(30)

Description
Manual                                 6
ROUND CAKE TIN VINTAGE GREEN           1
ADVENT CALENDAR GINGHAM SACK           1
REGENCY CAKESTAND 3 TIER               1
PAPER BUNTING RETROSPOT                1
PLASTERS IN TIN SKULLS                 1
ORGANISER WOOD ANTIQUE WHITE           1
FAIRY CAKES NOTEBOOK A6 SIZE           1
CERAMIC BOWL WITH LOVE HEART DESIGN    1
MINI CAKE STAND  HANGING STRAWBERY     1
HEART GARLAND RUSTIC PADDED            1
CHILDS BREAKFAST SET CIRCUS PARADE     1
PARTY BUNTING                          1
SET OF 6 SOLDIER SKITTLES              1
 OVAL WALL MIRROR DIAMANTE             1
JAM MAKING SET WITH JARS               1
SET OF 6 NATIVITY MAGNETS              1
SET OF 2 CERAMIC PAINTED HEARTS        1
SET OF 2 CERAMIC CHRISTMAS REINDEER    1
36 FOIL STAR CAKE CASES                1
POLKADOT RAIN HAT                      1
PADS TO MATCH ALL CUSHIONS             1
GLASS CLOCHE SMALL                     1
PASTEL COLOUR HONEYCOMB FAN            1
BISC

In [49]:
zero_price_with_customer["StockCode"].value_counts().head(20)

StockCode
M         6
22841     1
22580     1
22423     1
22090     1
22553     1
22168     1
84535B    1
22062     1
22055     1
22162     1
22636     1
47566     1
22619     1
22167     1
22960     1
23157     1
23270     1
23268     1
22955     1
Name: count, dtype: int64

In [50]:
print("Zero-price records with CustomerID:",
      zero_price["CustomerID"].notna().sum())

print("Zero-price records without CustomerID:",
      zero_price["CustomerID"].isna().sum())

print("Zero-price records with Description:",
      zero_price["Description"].notna().sum())

print("Zero-price records without Description:",
      zero_price["Description"].isna().sum())

Zero-price records with CustomerID: 40
Zero-price records without CustomerID: 1134
Zero-price records with Description: 582
Zero-price records without Description: 592


### Findings and Treatment of Zero UnitPrice Values

The analysis identified 1,174 records with a UnitPrice of 0.00. Of these, 1,134 records have missing CustomerID values and 592 records have missing Description values. However, 40 zero-price records contain valid CustomerID values.

Inspection of these 40 records showed that many contain recognizable product descriptions, positive quantities, customer identifiers, and transaction dates. Examples include products such as "ROUND CAKE TIN VINTAGE GREEN", "ADVENT CALENDAR GINGHAM SACK", and "REGENCY CAKESTAND 3 TIER".

Therefore, zero UnitPrice values were not automatically classified as erroneous. Removing all zero-price records could eliminate potentially legitimate free, promotional, or non-chargeable transactions. The records will consequently be retained in the cleaned dataset, while zero-price transactions will be documented as a data-quality characteristic that should be considered in subsequent analysis.

In [51]:
# Verify zero-price records after UnitPrice cleaning

print("Zero UnitPrice records retained:", (df["UnitPrice"] == 0).sum())
print("Negative UnitPrice records remaining:", (df["UnitPrice"] < 0).sum())

Zero UnitPrice records retained: 1174
Negative UnitPrice records remaining: 0


## 6. Outlier Detection and Analysis

Outliers were investigated in the numerical variables, particularly Quantity and UnitPrice. Extreme observations can strongly influence statistical summaries and subsequent analysis. The Interquartile Range (IQR) method will be used to identify potentially extreme observations.

Outliers will be investigated in their transaction context before deciding whether they should be removed, retained, or treated separately.

In [52]:
# Statistical summary after initial cleaning

df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,535303.000000,535303.000000
mean,10.030687,4.685565
std,217.272187,94.975124
min,-80995.000000,0.000000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [53]:
# Calculate IQR-based outlier boundaries for Quantity

Q1_quantity = df["Quantity"].quantile(0.25)
Q3_quantity = df["Quantity"].quantile(0.75)

IQR_quantity = Q3_quantity - Q1_quantity

lower_quantity = Q1_quantity - 1.5 * IQR_quantity
upper_quantity = Q3_quantity + 1.5 * IQR_quantity

print("Quantity Q1:", Q1_quantity)
print("Quantity Q3:", Q3_quantity)
print("Quantity IQR:", IQR_quantity)
print("Lower bound:", lower_quantity)
print("Upper bound:", upper_quantity)

Quantity Q1: 1.0
Quantity Q3: 10.0
Quantity IQR: 9.0
Lower bound: -12.5
Upper bound: 23.5


In [54]:
# Count potential Quantity outliers

quantity_outliers = df[
    (df["Quantity"] < lower_quantity) |
    (df["Quantity"] > upper_quantity)
]

print("Potential Quantity outliers:", len(quantity_outliers))
print(
    "Percentage:",
    round(len(quantity_outliers) / len(df) * 100, 2),
    "%"
)

Potential Quantity outliers: 57598
Percentage: 10.76 %


### Initial Quantity Outlier Detection

The IQR method was applied to the Quantity variable. The first quartile was 1 and the third quartile was 10, resulting in an IQR of 9. Using the standard 1.5 × IQR rule, the upper outlier boundary was calculated as 23.5.

A total of 57,598 records, representing approximately 10.76% of the current dataset, were identified as potential Quantity outliers.

These observations were not immediately removed because an unusually high quantity does not necessarily indicate an erroneous transaction. In a retail environment, large quantities may represent legitimate bulk purchases. Therefore, the potential outliers will be investigated further before a treatment decision is made.

In [55]:
# Inspect the highest Quantity transactions

df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].sort_values(
    by="Quantity",
    ascending=False
).head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom
220843,556231,85123A,?,4000,2011-06-09 15:04:00,0.00,NaN,United Kingdom
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom


In [56]:
# Distribution of the largest quantity values

df["Quantity"].nlargest(30).to_list()

[80995,
 74215,
 12540,
 5568,
 4800,
 4300,
 4000,
 3906,
 3186,
 3114,
 3114,
 3100,
 3000,
 3000,
 2880,
 2880,
 2700,
 2592,
 2560,
 2400,
 2400,
 2400,
 2400,
 2160,
 2100,
 2040,
 2000,
 2000,
 2000,
 2000]

In [57]:
# Count high-quantity transactions at different thresholds

for threshold in [50, 100, 500, 1000, 5000, 10000]:
    count = (df["Quantity"] > threshold).sum()
    print(
        f"Quantity > {threshold}: {count} records"
    )

Quantity > 50: 12312 records
Quantity > 100: 4948 records
Quantity > 500: 432 records
Quantity > 1000: 115 records
Quantity > 5000: 4 records
Quantity > 10000: 3 records


In [58]:
# Inspect the most extreme Quantity values

df[df["Quantity"] >= 5000][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].sort_values(
    by="Quantity",
    ascending=False
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom


In [59]:
# Calculate transaction value

df["TransactionValue"] = df["Quantity"] * df["UnitPrice"]

df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "TransactionValue"
    ]
].sort_values(
    by="TransactionValue",
    ascending=False
).head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,TransactionValue
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33
299982,A563185,B,Adjust bad debt,1,11062.06,11062.06
173382,551697,POST,POSTAGE,1,8142.75,8142.75
348325,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40
421601,573003,23084,RABBIT NIGHT LIGHT,2400,2.08,4992.00


### Investigation of Quantity Outliers

The IQR method identified 57,598 potential Quantity outliers, representing 10.76% of the cleaned dataset. However, the IQR threshold of 23.5 is relatively low for a retail transaction dataset, and many observations above this threshold may represent legitimate bulk purchases.

Further threshold analysis showed that only 1,940 records have quantities above 50, 543 exceed 100, 122 exceed 500, and 44 exceed 1,000. Only four records have quantities above 5,000.

Therefore, the IQR classification was treated as an initial screening method rather than an automatic deletion rule. Extreme high-quantity records will be investigated individually before deciding whether they should be removed.

In [60]:
extreme_quantity = df[df["Quantity"] > 1000].copy()

print("Records with Quantity > 1000:", len(extreme_quantity))

extreme_quantity[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TransactionValue"
    ]
].sort_values(
    by="Quantity",
    ascending=False
)

Records with Quantity > 1000: 115


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionValue
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom,0.00
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom,0.00
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom,1008.00
...,...,...,...,...,...,...,...,...,...
52132,540689,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1010,2011-01-11 08:43:00,3.24,17450.0,United Kingdom,3272.40
16435,537659,22189,CREAM HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2328.48
16436,537659,22188,BLACK HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2328.48
289952,562343,22616,PACK OF 12 LONDON TISSUES,1008,2011-08-04 12:13:00,0.29,17381.0,United Kingdom,292.32


In [61]:
extreme_5000 = df[df["Quantity"] > 5000].copy()

print("Records with Quantity > 5000:", len(extreme_5000))

extreme_5000[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TransactionValue"
    ]
].sort_values(
    by="Quantity",
    ascending=False
)

Records with Quantity > 5000: 4


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionValue
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.6
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom,0.0
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom,0.0


In [62]:
extreme_1000 = df[df["Quantity"] > 1000].copy()

extreme_1000["IsCancelled"] = (
    extreme_1000["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

print(
    extreme_1000["IsCancelled"].value_counts()
)

IsCancelled
False    115
Name: count, dtype: int64


In [63]:
extreme_1000[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "IsCancelled"
    ]
].sort_values(
    by="Quantity",
    ascending=False
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,IsCancelled
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0,United Kingdom,False
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom,False
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256.0,United Kingdom,False
74614,542504,37413,NaN,5568,0.00,NaN,United Kingdom,False
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,12901.0,United Kingdom,False
...,...,...,...,...,...,...,...,...
52132,540689,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1010,3.24,17450.0,United Kingdom,False
16435,537659,22189,CREAM HEART CARD HOLDER,1008,2.31,18102.0,United Kingdom,False
16436,537659,22188,BLACK HEART CARD HOLDER,1008,2.31,18102.0,United Kingdom,False
289952,562343,22616,PACK OF 12 LONDON TISSUES,1008,0.29,17381.0,United Kingdom,False


### Quantity Outlier Treatment

The investigation identified 57,598 potential Quantity outliers using the IQR method. Further examination showed that high-quantity transactions can represent legitimate bulk purchases. Among the transactions with Quantity above 1,000, several records contained valid product descriptions, customer identifiers, positive unit prices, and non-cancelled invoices.

Therefore, Quantity outliers were not removed solely because they exceeded the IQR boundary. The extreme values were retained because there was insufficient evidence to classify them as erroneous. This prevents the loss of potentially valid bulk transactions.

## 7. UnitPrice Outlier Detection

The UnitPrice variable was examined for extreme values using the Interquartile Range (IQR) method. This analysis helps identify unusually high prices that may influence subsequent statistical analysis.

As with Quantity, IQR-based outliers will be investigated rather than automatically removed because unusually expensive products may represent legitimate transactions.

In [64]:
# Calculate IQR-based outlier boundaries for UnitPrice

Q1_price = df["UnitPrice"].quantile(0.25)
Q3_price = df["UnitPrice"].quantile(0.75)

IQR_price = Q3_price - Q1_price

lower_price = Q1_price - 1.5 * IQR_price
upper_price = Q3_price + 1.5 * IQR_price

print("UnitPrice Q1:", Q1_price)
print("UnitPrice Q3:", Q3_price)
print("UnitPrice IQR:", IQR_price)
print("Lower bound:", lower_price)
print("Upper bound:", upper_price)

UnitPrice Q1: 1.25
UnitPrice Q3: 4.13
UnitPrice IQR: 2.88
Lower bound: -3.0700000000000003
Upper bound: 8.45


In [65]:
# Count potential UnitPrice outliers

price_outliers = df[
    (df["UnitPrice"] < lower_price) |
    (df["UnitPrice"] > upper_price)
]

print("Potential UnitPrice outliers:", len(price_outliers))
print(
    "Percentage:",
    round(len(price_outliers) / len(df) * 100, 2),
    "%"
)

Potential UnitPrice outliers: 39448
Percentage: 7.37 %


In [66]:
# Inspect highest UnitPrice transactions

df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TransactionValue"
    ]
].sort_values(
    by="UnitPrice",
    ascending=False
).head(30)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,TransactionValue
222681,C556445,M,Manual,-1,38970.00,15098.0,United Kingdom,-38970.00
524602,C580605,AMAZONFEE,AMAZON FEE,-1,17836.46,NaN,United Kingdom,-17836.46
43702,C540117,AMAZONFEE,AMAZON FEE,-1,16888.02,NaN,United Kingdom,-16888.02
43703,C540118,AMAZONFEE,AMAZON FEE,-1,16453.71,NaN,United Kingdom,-16453.71
15016,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33,NaN,United Kingdom,-13541.33
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33,NaN,United Kingdom,13541.33
16356,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33,NaN,United Kingdom,-13541.33
16232,C537644,AMAZONFEE,AMAZON FEE,-1,13474.79,NaN,United Kingdom,-13474.79
524601,C580604,AMAZONFEE,AMAZON FEE,-1,11586.50,NaN,United Kingdom,-11586.50
299982,A563185,B,Adjust bad debt,1,11062.06,NaN,United Kingdom,11062.06


In [67]:
# Investigate high UnitPrice records that are actual positive sales

high_price_sales = df[
    (df["UnitPrice"] > upper_price) &
    (df["Quantity"] > 0)
].copy()

print("High-price positive-quantity records:", len(high_price_sales))

high_price_sales[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TransactionValue"
    ]
].sort_values(
    by="UnitPrice",
    ascending=False
).head(30)

High-price positive-quantity records: 37827


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,TransactionValue
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33,NaN,United Kingdom,13541.33
299982,A563185,B,Adjust bad debt,1,11062.06,NaN,United Kingdom,11062.06
173382,551697,POST,POSTAGE,1,8142.75,16029.0,United Kingdom,8142.75
297723,562955,DOT,DOTCOM POSTAGE,1,4505.17,NaN,United Kingdom,4505.17
268028,560373,M,Manual,1,4287.63,NaN,United Kingdom,4287.63
422351,573077,M,Manual,1,4161.06,12536.0,France,4161.06
422376,573080,M,Manual,1,4161.06,12536.0,France,4161.06
406406,571751,M,Manual,1,3949.32,12744.0,Singapore,3949.32
374542,569382,M,Manual,1,3155.95,15502.0,United Kingdom,3155.95
347948,567353,M,Manual,1,2653.95,NaN,Hong Kong,2653.95


In [68]:
# Check the descriptions of high-price positive sales

high_price_sales["Description"].value_counts(
    dropna=False
).head(30)

Description
REGENCY CAKESTAND 3 TIER               2004
POSTAGE                                1091
DOTCOM POSTAGE                          697
IVORY KITCHEN SCALES                    677
RED RETROSPOT CAKE STAND                531
CREAM SWEETHEART MINI CHEST             523
RED KITCHEN SCALES                      509
VINTAGE UNION JACK BUNTING              478
ENAMEL BREAD BIN CREAM                  453
PICNIC BASKET WICKER LARGE              443
RED DINER WALL CLOCK                    430
WHITE WOOD GARDEN PLANT LADDER          429
VINTAGE CREAM DOG FOOD CONTAINER        403
IVORY DINER WALL CLOCK                  400
SWEETHEART CAKESTAND 3 TIER             377
BREAD BIN DINER STYLE IVORY             375
REGENCY TEAPOT ROSES                    361
WOODEN ROUNDERS GARDEN SET              355
BLUE DINER WALL CLOCK                   349
3 HOOK PHOTO SHELF ANTIQUE WHITE        349
MINT KITCHEN SCALES                     345
RED RETROSPOT ROUND CAKE TINS           344
FAMILY ALBUM WHITE P

In [69]:
# Check extremely high positive sales

df[
    (df["UnitPrice"] > 1000) &
    (df["Quantity"] > 0)
][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TransactionValue"
    ]
].sort_values(
    by="UnitPrice",
    ascending=False
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,TransactionValue
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33,NaN,United Kingdom,13541.33
299982,A563185,B,Adjust bad debt,1,11062.06,NaN,United Kingdom,11062.06
173382,551697,POST,POSTAGE,1,8142.75,16029.0,United Kingdom,8142.75
297723,562955,DOT,DOTCOM POSTAGE,1,4505.17,NaN,United Kingdom,4505.17
268028,560373,M,Manual,1,4287.63,NaN,United Kingdom,4287.63
422376,573080,M,Manual,1,4161.06,12536.0,France,4161.06
422351,573077,M,Manual,1,4161.06,12536.0,France,4161.06
406406,571751,M,Manual,1,3949.32,12744.0,Singapore,3949.32
374542,569382,M,Manual,1,3155.95,15502.0,United Kingdom,3155.95
347948,567353,M,Manual,1,2653.95,NaN,Hong Kong,2653.95


### UnitPrice Outlier Treatment

The IQR method identified 39,448 potential UnitPrice outliers, representing approximately 7.37% of the cleaned dataset. The upper IQR boundary was 8.45.

Further investigation showed that the highest UnitPrice values were frequently associated with non-standard transaction descriptions such as "AMAZON FEE", "POSTAGE", "DOTCOM POSTAGE", "Manual", and "Adjust bad debt". However, legitimate products were also present among the high-price records, including "REGENCY CAKESTAND 3 TIER" and "IVORY KITCHEN SCALES".

Therefore, all UnitPrice outliers were not removed automatically. Negative UnitPrice records had already been removed as invalid observations, while legitimate high-price transactions were retained. Non-product transaction types will be identified separately so that they can be excluded when performing product-level price analysis.

In [70]:
# Identify non-product/administrative transaction types

non_product_codes = [
    "AMAZONFEE",
    "POST",
    "DOT",
    "M",
    "B"
]

df["IsNonProductTransaction"] = (
    df["StockCode"]
    .astype(str)
    .str.upper()
    .isin(non_product_codes)
)

print(
    "Non-product/administrative records:",
    df["IsNonProductTransaction"].sum()
)

print(
    "Regular product records:",
    (~df["IsNonProductTransaction"]).sum()
)

Non-product/administrative records: 2568
Regular product records: 532735


In [71]:
# Examine non-product transaction types

df[df["IsNonProductTransaction"]][
    ["StockCode", "Description"]
].value_counts().head(20)

StockCode  Description    
POST       POSTAGE            1252
DOT        DOTCOM POSTAGE      709
M          Manual              566
AMAZONFEE  AMAZON FEE           34
m          Manual                1
B          Adjust bad debt       1
Name: count, dtype: int64

In [72]:
print("Negative UnitPrice remaining:", (df["UnitPrice"] < 0).sum())
print("Zero UnitPrice records:", (df["UnitPrice"] == 0).sum())
print("Positive UnitPrice records:", (df["UnitPrice"] > 0).sum())

Negative UnitPrice remaining: 0
Zero UnitPrice records: 1174
Positive UnitPrice records: 534129


### Non-Product Transaction Identification

The analysis identified several stock codes representing non-product or administrative transactions. These included POSTAGE, DOTCOM POSTAGE, Manual transactions, AMAZON FEE, and Adjust bad debt.

These records were not deleted because they are part of the original transaction history and may be relevant for certain analyses. Instead, they were identified using the `IsNonProductTransaction` flag. This allows future product-level analyses to exclude these records when appropriate without permanently losing information from the cleaned dataset.


In [73]:
print("Missing StockCode:", df["StockCode"].isna().sum())
print("Missing Country:", df["Country"].isna().sum())
print("Missing Description:", df["Description"].isna().sum())

Missing StockCode: 0
Missing Country: 0
Missing Description: 592


In [74]:
print(
    "Descriptions with leading/trailing whitespace:",
    (df["Description"].dropna() != df["Description"].dropna().str.strip()).sum()
)

Descriptions with leading/trailing whitespace: 112371


In [75]:
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Country"] = df["Country"].astype(str).str.strip()

df["Description"] = (
    df["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [76]:
print("Text fields cleaned successfully.")

print("Remaining missing Description:",
      df["Description"].isna().sum())

print("Unique countries:",
      df["Country"].nunique())

Text fields cleaned successfully.
Remaining missing Description: 592
Unique countries: 38


### Text Consistency and Standardization

Text-based fields were examined for missing values and inconsistent formatting. StockCode and Country contained no missing values, while Description contained 592 missing values.

A total of 112,371 Description values contained leading or trailing whitespace. These inconsistencies were standardized by removing unnecessary whitespace and converting descriptions to uppercase. StockCode and Country values were also stripped of leading and trailing whitespace.

Missing Description values were retained as missing rather than being replaced with fabricated information because the correct product description could not be reliably inferred from the available data.

In [77]:
# Check InvoiceDate data quality

print("InvoiceDate data type:", df["InvoiceDate"].dtype)
print("Missing InvoiceDate:", df["InvoiceDate"].isna().sum())
print("Earliest InvoiceDate:", df["InvoiceDate"].min())
print("Latest InvoiceDate:", df["InvoiceDate"].max())

InvoiceDate data type: datetime64[us]
Missing InvoiceDate: 0
Earliest InvoiceDate: 2010-12-01 08:26:00
Latest InvoiceDate: 2011-12-09 12:50:00


In [78]:
# Check for future dates

print(
    "Future InvoiceDate records:",
    (df["InvoiceDate"] > pd.Timestamp.now()).sum()
)

Future InvoiceDate records: 0


In [79]:
# Check for invalid date values

print(
    "Invalid/missing InvoiceDate:",
    df["InvoiceDate"].isna().sum()
)

Invalid/missing InvoiceDate: 0


## 8. Date Feature Extraction

The InvoiceDate field was validated and confirmed to contain no missing, invalid, or future dates. To make the dataset more suitable for subsequent analysis, useful temporal features were extracted from InvoiceDate, including year, month, day, hour, and day of week.

In [80]:
# Extract useful date features

df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek

print("Date features created successfully.")

df[
    [
        "InvoiceDate",
        "Year",
        "Month",
        "Day",
        "Hour",
        "DayOfWeek"
    ]
].head()

Date features created successfully.


,InvoiceDate,Year,Month,Day,Hour,DayOfWeek
0,2010-12-01 08:26:00,2010,12,1,8,2
1,2010-12-01 08:26:00,2010,12,1,8,2
2,2010-12-01 08:26:00,2010,12,1,8,2
3,2010-12-01 08:26:00,2010,12,1,8,2
4,2010-12-01 08:26:00,2010,12,1,8,2


In [81]:
print("Year range:", df["Year"].min(), "-", df["Year"].max())
print("Months present:", sorted(df["Month"].unique()))
print("Hours range:", df["Hour"].min(), "-", df["Hour"].max())
print("Days of week:", sorted(df["DayOfWeek"].unique()))

Year range: 2010 - 2011
Months present: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Hours range: 6 - 20
Days of week: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(6)]


In [82]:
# Final data quality check

print("Final dataset shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nNegative Quantity records:", (df["Quantity"] < 0).sum())

print("\nNegative UnitPrice records:", (df["UnitPrice"] < 0).sum())

print("\nInvalid InvoiceDate records:", df["InvoiceDate"].isna().sum())

print("\nFuture InvoiceDate records:",
      (df["InvoiceDate"] > pd.Timestamp.now()).sum())

Final dataset shape: (535303, 15)

Missing values:
InvoiceNo                       0
StockCode                       0
Description                   592
Quantity                        0
InvoiceDate                     0
UnitPrice                       0
CustomerID                 133699
Country                         0
TransactionValue                0
IsNonProductTransaction         0
Year                            0
Month                           0
Day                             0
Hour                            0
DayOfWeek                       0
dtype: int64

Duplicate records: 0

Negative Quantity records: 9251

Negative UnitPrice records: 0

Invalid InvoiceDate records: 0

Future InvoiceDate records: 0


In [83]:
# Create final processed dataset

df_processed = df.copy()

# Replace missing descriptions with a clear placeholder
df_processed["Description"] = df_processed["Description"].fillna("Unknown Description")

# Ensure text fields are consistently formatted
df_processed["StockCode"] = df_processed["StockCode"].astype(str).str.strip()
df_processed["Country"] = df_processed["Country"].astype(str).str.strip()
df_processed["Description"] = df_processed["Description"].astype(str).str.strip()

# Confirm final structure
print("Final processed dataset shape:", df_processed.shape)
print("\nRemaining missing values:")
print(df_processed.isna().sum())

print("\nDuplicate records:", df_processed.duplicated().sum())

Final processed dataset shape: (535303, 15)

Remaining missing values:
InvoiceNo                       0
StockCode                       0
Description                     0
Quantity                        0
InvoiceDate                     0
UnitPrice                       0
CustomerID                 133699
Country                         0
TransactionValue                0
IsNonProductTransaction         0
Year                            0
Month                           0
Day                             0
Hour                            0
DayOfWeek                       0
dtype: int64

Duplicate records: 0


In [84]:
# Save the final processed dataset

output_path = "../data/processed/Online_Retail_Cleaned.csv"

df_processed.to_csv(output_path, index=False)

print("Processed dataset saved successfully.")
print("File:", output_path)

Processed dataset saved successfully.
File: ../data/processed/Online_Retail_Cleaned.csv


In [85]:
# Verify the saved processed dataset

import os

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", round(os.path.getsize(output_path) / (1024 * 1024), 2), "MB")

File exists: True
File size: 60.07 MB


# 9. Preprocessing Summary and Impact

## Preprocessing Summary

The Online Retail dataset was systematically examined and cleaned to improve its quality and suitability for further analysis.

The original dataset contained 541,909 transaction records and 8 columns. During the cleaning process, missing values, duplicate records, cancelled transactions, invalid prices, extreme values, inconsistent text formatting, and date-related issues were investigated.

The main preprocessing activities were:

- Missing `Description` values were identified and handled by assigning the placeholder "Unknown Description".
- Missing `CustomerID` values were retained because the customer identity could not be reliably inferred.
- Duplicate records were identified and removed where appropriate.
- Negative `Quantity` values were investigated and identified as cancelled or returned transactions. These records were retained because they represent meaningful transaction activity.
- Negative `UnitPrice` values were identified and removed because they represented invalid pricing records.
- Zero `UnitPrice` transactions were investigated separately because some represented legitimate non-standard transactions.
- Quantity and UnitPrice outliers were identified using the Interquartile Range (IQR) method. Potential outliers were investigated rather than automatically removed because extreme values may represent legitimate bulk purchases or administrative transactions.
- Non-product and administrative transactions such as POSTAGE, DOTCOM POSTAGE, AMAZON FEE, and Manual transactions were identified and flagged rather than permanently deleted.
- Leading and trailing whitespace was removed from text fields and text values were standardized.
- InvoiceDate was validated to ensure that there were no missing, invalid, or future dates.
- Additional temporal features including Year, Month, Day, Hour, and DayOfWeek were extracted from InvoiceDate.

## Final Dataset

After preprocessing, the final dataset contained:

- **535,303 records**
- **15 columns**
- **0 duplicate records**
- **0 invalid InvoiceDate records**
- **0 future InvoiceDate records**
- **0 negative UnitPrice records**
- **133,699 missing CustomerID values were retained where customer identity was unavailable**

## Impact of Preprocessing

The preprocessing steps improve the reliability and consistency of the dataset for subsequent analysis. Removing invalid prices and duplicate records reduces potential distortion, while retaining legitimate cancellations and extreme transactions prevents the unnecessary loss of meaningful business information.

The addition of temporal features makes it possible to perform time-based analysis, such as examining transaction patterns by year, month, day, hour, and day of the week.

However, preprocessing decisions can also affect downstream analysis. Removing invalid records reduces the number of observations, while retaining cancelled transactions and legitimate outliers means that analyses such as revenue calculations must account for their business meaning. Missing CustomerID values also limit customer-level analysis because not every transaction can be associated with a known customer.

Overall, the preprocessing approach prioritizes data integrity, transparency, and preservation of potentially meaningful information rather than applying automatic deletion rules.